## ÉTAPE 1 – Configuration de l'environnement

In [ ]:
# Importer les librairies nécessaires
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *
import snowflake.snowpark as snowpark

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Obtenir la session Snowflake active (dans un Notebook Snowflake)
session = get_active_session()
print(' Session Snowflake active :', session.get_current_database())

In [ ]:
# Configuration de l'environnement
session.sql('USE DATABASE MY_GRP_LAB').collect()
session.sql('USE WAREHOUSE MY_GRP_WH').collect()

# Créer le schéma ML si inexistant
session.sql('CREATE SCHEMA IF NOT EXISTS ML').collect()
session.sql('USE SCHEMA ML').collect()

print('Environnement configuré')
print('Database :', session.get_current_database())
print('Schema   :', session.get_current_schema())
print('Warehouse:', session.get_current_warehouse())

## ÉTAPE 2 – Ingestion des données depuis S3

In [ ]:
# Créer le stage S3
session.sql("""
    CREATE STAGE IF NOT EXISTS S3_HOUSE_PRICE
        URL = 's3://logbrain-datalake/datasets/house_price/'
        FILE_FORMAT = (
            TYPE = 'CSV'
            FIELD_OPTIONALLY_ENCLOSED_BY = '"'
            SKIP_HEADER = 1
            NULL_IF = ('', 'NULL')
            EMPTY_FIELD_AS_NULL = TRUE
            TRIM_SPACE = TRUE
        )
""").collect()

# Vérifier les fichiers disponibles
files = session.sql('LIST @S3_HOUSE_PRICE').collect()
print('Fichiers disponibles dans S3 :')
for f in files:
    print(' -', f[0])

In [ ]:
# Créer la table BRONZE
session.sql("""
    CREATE TABLE IF NOT EXISTS ML.HOUSE_PRICE_RAW (
        price            NUMBER,
        area             NUMBER,
        bedrooms         NUMBER,
        bathrooms        NUMBER,
        stories          NUMBER,
        mainroad         VARCHAR(10),
        guestroom        VARCHAR(10),
        basement         VARCHAR(10),
        hotwaterheating  VARCHAR(10),
        airconditioning  VARCHAR(10),
        parking          NUMBER,
        prefarea         VARCHAR(10),
        furnishingstatus VARCHAR(20)
    )
""").collect()

# Charger les données
session.sql("""
    COPY INTO ML.HOUSE_PRICE_RAW
    FROM @S3_HOUSE_PRICE
    FILE_FORMAT = (
        TYPE = 'CSV'
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        SKIP_HEADER = 1
        EMPTY_FIELD_AS_NULL = TRUE
        TRIM_SPACE = TRUE
    )
    ON_ERROR = 'CONTINUE'
""").collect()

# Vérifier le chargement
count = session.sql('SELECT COUNT(*) AS nb FROM ML.HOUSE_PRICE_RAW').collect()[0][0]
print(f'{count} lignes chargées dans ML.HOUSE_PRICE_RAW')

#ÉTAPE 3 – Exploration des données

In [ ]:
# ÉTAPE 3.1 — Charger les données depuis Snowflake vers pandas
df = session.table('MY_GRP_LAB.ML.HOUSE_PRICE_RAW').to_pandas()

print(' Dimensions du dataset :', df.shape)
print('\n Aperçu des 10 premières lignes :')
df.head(10)

In [ ]:
# ÉTAPE 3.2 — Statistiques descriptives
print(' Statistiques descriptives :')
df.describe()

In [ ]:
# ÉTAPE 3.3 — Détecter les valeurs manquantes
print(' Valeurs manquantes par colonne :')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else ' Aucune valeur manquante')

In [ ]:
# ÉTAPE 3.4 — Distribution du prix (variable cible)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme du prix brut
axes[0].hist(df['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du Prix')
axes[0].set_xlabel('Prix ($)')
axes[0].set_ylabel('Fréquence')

# Histogramme du log(prix)
axes[1].hist(np.log(df['PRICE']), bins=30, color='coral', edgecolor='white')
axes[1].set_title('Distribution du Log(Prix)')
axes[1].set_xlabel('Log Prix')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# ÉTAPE 3.5 — Matrice de corrélation entre variables numériques
numeric_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5
)
plt.title('Matrice de Corrélation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ÉTAPE 3.6 — Impact des variables catégorielles sur le prix
cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'AIRCONDITIONING',
            'PREFAREA', 'FURNISHINGSTATUS']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df.groupby(col)['PRICE'].mean().plot(
        kind='bar', ax=axes[i],
        color='steelblue', edgecolor='white'
    )
    axes[i].set_title(f'Prix moyen par {col}')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

ETAPE4 - Préparation des features

In [ ]:
# ÉTAPE 4.1 — Encodage des variables catégorielles
# Convertir yes/no en 1/0 (les modèles ML ne comprennent que les chiffres)
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
               'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']

for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Encodage ordinal pour furnishingstatus (3 niveaux : 0, 1, 2)
df['FURNISHINGSTATUS'] = df['FURNISHINGSTATUS'].map({
    'furnished':      2,
    'semi-furnished': 1,
    'unfurnished':    0
})

print(' Encodage terminé')
df.head()

In [ ]:
# ÉTAPE 4.2 — Séparation train/test + Normalisation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Définir features (X) et cible (y)
FEATURES = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'MAINROAD',
            'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 'AIRCONDITIONING',
            'PARKING', 'PREFAREA', 'FURNISHINGSTATUS']
TARGET = 'PRICE'

X = df[FEATURES]
y = df[TARGET]

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalisation (StandardScaler : moyenne=0, écart-type=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f' Split effectué :')
print(f'   Train : {X_train.shape[0]} lignes')
print(f'   Test  : {X_test.shape[0]} lignes')
print(f'   Features : {X_train.shape[1]} variables')

In [ ]:
# ÉTAPE 4.3 — Sauvegarder les données préparées dans Snowflake
df_prepared = df[FEATURES + [TARGET]].copy()

session.write_pandas(
    df_prepared,
    table_name='HOUSE_PRICE_PREPARED',
    database='MY_GRP_LAB',
    schema='ML',
    overwrite=True,
    auto_create_table=True
)
print(' Données préparées sauvegardées dans MY_GRP_LAB.ML.HOUSE_PRICE_PREPARED')
print(f'   Nombre de lignes : {len(df_prepared)}')
print(f'   Nombre de colonnes : {len(df_prepared.columns)}')

In [ ]:
# Vérification finale : la table est bien créée et accessible
verify = session.sql('SELECT COUNT(*) AS nb FROM MY_GRP_LAB.ML.HOUSE_PRICE_PREPARED').collect()
print(f' Vérification : {verify[0][0]} lignes dans la table HOUSE_PRICE_PREPARED')

# Aperçu des données préparées
session.table('MY_GRP_LAB.ML.HOUSE_PRICE_PREPARED').to_pandas().head(5)

Etape 5 et 6 - Entrainement et evaluation des modèles

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Entraîne et évalue un modèle, retourne les métriques"""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2   = r2_score(y_te, y_pred)

    return {
        'Modèle': name,
        'MAE':    round(mae, 2),
        'RMSE':   round(rmse, 2),
        'R²':     round(r2, 4)
    }

# Définir les modèles à tester
models = {
    'Linear Regression':     LinearRegression(),
    'Ridge Regression':      Ridge(alpha=1.0),
    'Random Forest':         RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':     GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':               xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
}

# Entraîner et évaluer tous les modèles
results = []
trained_models = {}

for name, model in models.items():
    print(f'⏳ Entraînement : {name}...')
    result = evaluate_model(name, model, X_train_scaled, X_test_scaled, y_train, y_test)
    results.append(result)
    trained_models[name] = model
    print(f'    R² = {result["R²"]} | MAE = {result["MAE"]:,.0f} | RMSE = {result["RMSE"]:,.0f}')

# Tableau de comparaison
df_results = pd.DataFrame(results).sort_values('R²', ascending=False)
print('\n Comparaison des modèles :')
df_results

In [ ]:
# Visualisation des performances
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['R²', 'MAE', 'RMSE']
colors  = ['steelblue', 'coral', 'green']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ascending = metric != 'R²'
    df_plot = df_results.sort_values(metric, ascending=ascending)
    axes[i].barh(df_plot['Modèle'], df_plot[metric], color=color, edgecolor='white')
    axes[i].set_title(f'Comparaison – {metric}')
    axes[i].set_xlabel(metric)

plt.tight_layout()
plt.show()

# Meilleur modèle
best_model_name = df_results.iloc[0]['Modèle']
print(f'\n Meilleur modèle : {best_model_name} (R² = {df_results.iloc[0]["R²"]})')

Etape 7 - Optimisation avec Hyperparamètres

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Optimisation du meilleur modèle (XGBoost ou Random Forest)
print(' Optimisation des hyperparamètres avec GridSearchCV...')

# Paramètres à tester pour XGBoost
param_grid_xgb = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

xgb_model = xgb.XGBRegressor(random_state=42, verbosity=0)

# RandomizedSearchCV (plus rapide que GridSearch)
random_search = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_grid_xgb,
    n_iter=20,
    scoring='r2',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_scaled, y_train)

print('\n Meilleurs paramètres trouvés :')
print(random_search.best_params_)
print(f'   Meilleur R² (CV) : {random_search.best_score_:.4f}')

In [ ]:
# Évaluer le modèle optimisé sur le test set
best_model = random_search.best_estimator_
y_pred_best = best_model.predict(X_test_scaled)

mae_best  = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best   = r2_score(y_test, y_pred_best)

print(' Performance du modèle optimisé :')
print(f'   R²   : {r2_best:.4f}')
print(f'   MAE  : {mae_best:,.0f}')
print(f'   RMSE : {rmse_best:,.0f}')

# Visualisation prédictions vs réalité
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, color='steelblue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Prix réel')
plt.ylabel('Prix prédit')
plt.title(f'Prédictions vs Réalité (R² = {r2_best:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features
feature_importance = pd.DataFrame({
    'Feature':   FEATURES,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'],
         feature_importance['Importance'],
         color='steelblue', edgecolor='white')
plt.title('Importance des Features – Modèle Optimisé')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## ÉTAPE 8 – Stockage dans le Model Registry

In [ ]:
from snowflake.ml.registry import Registry
import sklearn

# Créer le Model Registry
registry = Registry(session=session, database_name='MY_GRP_LAB', schema_name='ML')
print('Model Registry initialisé')

In [ ]:
from snowflake.ml.model import custom_model

# Enregistrer le meilleur modèle dans le registry
model_version = registry.log_model(
    model=best_model,
    model_name='HOUSE_PRICE_PREDICTOR',
    version_name='V1',
    comment='XGBoost optimisé avec RandomizedSearchCV – MBA ESG 2026',
    metrics={
        'r2':   round(r2_best, 4),
        'mae':  round(mae_best, 2),
        'rmse': round(rmse_best, 2)
    },
    sample_input_data=pd.DataFrame(X_test_scaled[:5], columns=FEATURES)
)

print('Modèle enregistré dans le Snowflake Model Registry !')
print(f'   Nom    : HOUSE_PRICE_PREDICTOR')
print(f'   Version: V1')
print(f'   R²     : {r2_best:.4f}')

In [ ]:
# Lister les modèles enregistrés
print('modèles dans le Registry :')
models_list = registry.show_models()
print(models_list)

## ÉTAPE 9 – Inférence avec le modèle


In [ ]:
# Charger le modèle depuis le Registry
loaded_model = registry.get_model('HOUSE_PRICE_PREDICTOR').version('V1')
print('modèle chargé depuis le Registry')

In [ ]:
# Inférence sur les données de test
test_df = pd.DataFrame(X_test_scaled, columns=FEATURES)
predictions = loaded_model.run(test_df, function_name='predict')

# Ajouter les vraies valeurs pour comparaison
results_df = test_df.copy()
results_df['PRICE_REEL']   = y_test.values
results_df['PRICE_PREDIT'] = predictions
results_df['ERREUR_PCT']   = abs(
    results_df['PRICE_REEL'] - results_df['PRICE_PREDIT']
) / results_df['PRICE_REEL'] * 100

print('résultats des prédictions :')
print(f'   Erreur moyenne : {results_df["ERREUR_PCT"].mean():.1f}%')
results_df[['PRICE_REEL', 'PRICE_PREDIT', 'ERREUR_PCT']].head(10)

In [ ]:
# Sauvegarder les prédictions dans Snowflake
session.write_pandas(
    results_df[['PRICE_REEL', 'PRICE_PREDIT', 'ERREUR_PCT']],
    table_name='HOUSE_PRICE_PREDICTIONS',
    database='MY_GRP_LAB',
    schema='ML',
    overwrite=True
)
print('Prédictions sauvegardées dans ML.HOUSE_PRICE_PREDICTIONS')

## ÉTAPE 10 – Application Streamlit
**Voir fichier : streamlit_app.py**